# Tooltip `group`

`layer_tooltips().group()` controls which layers participate in the same tooltip. Line-like geoms are merged automatically, and `group()` lets you override that behavior when needed or merge layers that would otherwise stay separate.

In [ ]:
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

In [ ]:
data = pd.DataFrame({
    'x': [1, 2, 3, 4, 5],
    'y1': [13, 15, 14, 18, 17],
    'y2': [1, 2, 5, 4, 6],
    'y3': [5.5, 10, 7, 10.5, 13]
})

metric_df = pd.DataFrame({
    'week': [1, 2, 3, 4, 5, 6, 7, 8],
    'value': [13, 15, 14, 18, 17, 16, 19, 18]
})

events_df = pd.DataFrame({
    'week': [2, 5, 7],
    'value': [15, 17, 19],
    'event': ['Release 1.2', 'Campaign launch', 'Incident fixed']
})

## Automatic merging of line-like layers

Line-like geoms are merged by default. This example uses two `geom_line()` layers and one `geom_smooth()` layer; they all contribute to the same tooltip without explicit `group()`.

In [ ]:
ggplot(data, aes(x='x')) \
    + geom_line(
        aes(y='y1'),
        tooltips=layer_tooltips()
            .format('@y1', '.1f')
            .line('Series A: @y1')
    ) \
    + geom_line(
        aes(y='y2'),
        tooltips=layer_tooltips()
            .format('@y2', '.2f')
            .line('Series B: @y2')
    ) \
    + geom_smooth(
        aes(y='y3'),
        method='loess',
        se=False,
        tooltips=layer_tooltips()
            .format('@y3', '.2f')
            .line('Series C: @y3')
    )

## Split same-kind layers

Two `geom_line()` layers are merged by default. Give them different tooltip groups when they represent different features and should not expose both tooltips at once.

In [ ]:
ggplot(data, aes(x='x')) \
    + geom_line(
        aes(y='y1'),
        tooltips=layer_tooltips()
            .group('series_a')
            .format('@y1', '.1f')
            .line('Series A: @y1')
    ) \
    + geom_line(
        aes(y='y2'),
        tooltips=layer_tooltips()
            .group('series_b')
            .format('@y2', '.2f')
            .line('Series B: @y2')
    )

## Merge layers with different lookup policy

A common use case is a time series plus a separate event table: the line shows the metric, while points mark releases, campaigns, or incidents. A shared tooltip group lets both layers contribute to one tooltip.

In [ ]:
policy_default = (
    ggplot(metric_df, aes(x='week'))
    + geom_line(
        aes(y='value'),
        tooltips=layer_tooltips()
            .format('@value', '.1f')
            .line('Metric: @value')
    )
    + geom_point(
        aes(x='week', y='value'),
        data=events_df,
        tooltips=layer_tooltips()
            .line('Event: @event')
    )
    + ggtitle('Default')
)

policy_grouped = (
    ggplot(metric_df, aes(x='week'))
    + geom_line(
        aes(y='value'),
        tooltips=layer_tooltips()
            .group('metric')
            .format('@value', '.1f')
            .line('Metric: @value')
    )
    + geom_point(
        aes(x='week', y='value'),
        data=events_df,
        tooltips=layer_tooltips()
            .group('metric')
            .line('Event: @event')
    )
    + ggtitle('Same `group`: merged')
)

gggrid([policy_default, policy_grouped], ncol=2)